# Knowledge Graphs & Graph Neural Networks in AEC
### Annotated Learning Notebook

---

> **Purpose of this notebook:** This is an annotated, instructional version of `KG_GNN_AEC.ipynb`. Each section includes `📌 Instruction` notes that explain *what* is happening, *why* it matters, and *how* the pieces connect. Run the original notebook first, or use this one as a guided reading companion.

---

## Big Picture

This notebook explores a complete pipeline for applying **Knowledge Graphs (KG)** and **Graph Neural Networks (GNN)** to Architecture, Engineering and Construction (AEC) data.

The pipeline has four stages:

| Stage | Topic | Key tool |
|-------|-------|----------|
| 1 | Graph theory fundamentals | `networkx` |
| 2 | Building a Knowledge Graph from AEC data | `rdflib`, Real Estate Core ontology |
| 3 | Predicting room adjacency with a GNN | `PyTorch Geometric` |
| 4 | Querying the KG with natural language (Graph RAG) | `OpenAI` + SPARQL |

**Domain context:** In AEC, floor plans define which rooms are spatially adjacent. When data is incomplete (e.g., a PDF floor plan without adjacency tags), we need to *infer* those connections automatically. That is the core prediction task.

---
## Part 0 — Setup

### 📌 Instruction: Installing dependencies (Google Colab only)

The original notebook targets **Google Colab**. The install block below uses `%%capture` to suppress noisy pip output. The key libraries are:

- `torch-scatter`, `torch-sparse`, `torch-geometric` — sparse-graph operations for GNNs
- `rdflib` — building and querying RDF Knowledge Graphs
- `yfiles_jupyter_graphs` — interactive graph visualization inside Jupyter
- `oxigraph` — fast in-memory RDF store (used as backend for rdflib)
- `gdown` — downloads files from Google Drive

If running **locally**, install via pip (match your CUDA/CPU torch version):
```bash
pip install torch-geometric rdflib yfiles-jupyter-graphs oxigraph gdown openai
```

In [ ]:
# ── Colab only: uncomment if running on Google Colab ──────────────────────────
# %%capture
# import os, torch
# os.environ['TORCH'] = torch.__version__
# !pip install torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-${TORCH}.html
# !pip install rdflib yfiles-jupyter-graphs oxigraph gdown openai
# ──────────────────────────────────────────────────────────────────────────────

### 📌 Instruction: Core imports

Key libraries by role:

| Library | Role |
|---------|------|
| `networkx` | General-purpose graph manipulation and visualization |
| `rdflib` | Parse, build, and SPARQL-query RDF Knowledge Graphs |
| `torch` + `torch_geometric` | Define and train Graph Neural Networks |
| `shapely` | Geometric operations on room polygons |
| `plotly` | Interactive visualization |
| `openai` | LLM calls for Graph RAG section |

In [ ]:
import torch
import matplotlib as mpl
import pickle
import numpy as np
import networkx as nx
from yfiles_jupyter_graphs import GraphWidget
from functools import reduce
from typing import Dict, List, Union, Tuple
from pprint import pprint
import rdflib
from rdflib import Graph, Literal, RDF, URIRef, Namespace
from hashlib import md5
import re
from shapely import wkt
import plotly.graph_objs as go
import pandas as pd
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F
import tqdm, gdown, os, io
from openai import OpenAI
from itertools import combinations
from copy import copy

if not os.path.exists("data"):
    os.mkdir("data")

print("Imports OK")

---
## Part 1 — Graph Theory Fundamentals

### 📌 Instruction: What is a graph?

A **graph** $G = (V, E)$ consists of:
- **Vertices (nodes)** $V$ — the objects (e.g., rooms, buildings, sensors)
- **Edges** $E$ — the relationships (e.g., *room A is adjacent to room B*)

In AEC, almost everything is relational:
- A building has floors → floors have apartments → apartments have rooms → rooms are adjacent to each other
- These hierarchies are naturally modelled as graphs.

We use `networkx` to create and visualise graphs. The **adjacency matrix** is a square matrix where `A[i][j] = 1` means nodes `i` and `j` are connected.

In [ ]:
# Define 7 nodes and 10 edges forming a simple undirected graph
nodes = [1, 2, 3, 4, 5, 6, 7]
edges = [(1,2),(1,3),(1,7),(2,3),(3,4),(3,7),(4,6),(4,5),(5,6),(6,7)]

g = nx.Graph()          # undirected: (A,B) == (B,A)
g.add_nodes_from(nodes)
g.add_edges_from(edges)

# The adjacency matrix encodes connectivity as a 2D array
adjacency_matrix = nx.to_numpy_array(g)
print("Adjacency Matrix (7x7):")
print(adjacency_matrix)
# Each row/column corresponds to a node; 1.0 means connected, 0.0 means not connected

### 📌 Instruction: Graph types relevant to AEC

| Type | Key property | AEC example |
|------|-------------|-------------|
| **Undirected** | $(A,B) = (B,A)$ | Room adjacency (symmetric) |
| **Directed** | $(A,B) \neq (B,A)$ | Air flow direction, drainage |
| **Multigraph** | Multiple edges between same pair | A wall that is both structural *and* acoustic |
| **Directed Multigraph** | Both above | IFC relationships (most AEC KGs use this) |

The **Directed Multigraph** is the most expressive — and it is the base for Knowledge Graphs.

In [ ]:
# A directed multigraph allows parallel edges with different meanings
multidigraph = nx.MultiDiGraph()
multidigraph.add_nodes_from(nodes)

# Multiple directed edges between the same pair of nodes
multidigraph.add_edges_from([
    (1, 2, {"type": "structuralSupport"}),
    (1, 2, {"type": "acousticBoundary"}),   # second edge, different relation
    (2, 3, {"type": "structuralSupport"}),
])

print(f"Nodes: {list(multidigraph.nodes)}")
print(f"Edges: {list(multidigraph.edges(data=True))}")

---
## Part 2 — Homogeneous vs Heterogeneous Graphs

### 📌 Instruction: Why heterogeneous graphs matter in AEC

A **homogeneous** graph has one type of node and one type of edge.

A **heterogeneous** graph $G = (V, E, R, T)$ extends this with:
- Multiple **node types** $T$ (e.g., Building, Floor, Room, Sensor)
- Multiple **edge/relation types** $R$ (e.g., `hasPart`, `adjacentTo`, `hasGeometry`)

Real AEC data is almost always heterogeneous — an IFC file contains walls, slabs, spaces, and doors that are all different types of objects with different properties.

**Why does this distinction matter for ML?** Standard GNNs assume one node type. When you have multiple types, you need *heterogeneous* GNNs (like the one used in Part 4).

In [ ]:
# Homogeneous graph: all nodes have the same type "A"
g1 = g.to_directed().copy()
nx.set_node_attributes(g1, {n: {"type": "A"} for n in g1.nodes()})
nx.set_edge_attributes(g1, {e: {"type": "relation"} for e in g1.edges()})
print("Homogeneous graph — all nodes are type 'A'")
print("Node types:", set(nx.get_node_attributes(g1, 'type').values()))

# Heterogeneous graph: even-numbered nodes are type "A", odd are type "B"
g2 = multidigraph.copy()
for n in g2.nodes():
    g2.nodes[n]['type'] = 'A' if n % 2 == 0 else 'B'
print("\nHeterogeneous graph — nodes are type 'A' or 'B'")
print("Node types:", set(nx.get_node_attributes(g2, 'type').values()))

---
## Part 3 — Knowledge Graphs

### 📌 Instruction: What is a Knowledge Graph?

A **Knowledge Graph (KG)** is a heterogeneous directed multigraph that encodes factual knowledge:

- It uses the **RDF (Resource Description Framework)** data model
- Everything is expressed as **triples**: `<subject> <predicate> <object>`
- Example: `<Room_101> <adjacentTo> <Room_102>`
- Entities are identified by **URIs** (Universal Resource Identifiers), ensuring uniqueness globally

### RDF Triple Structure

```
Subject ──── Predicate ───► Object
  │                           │
 URI                     URI or Literal
(entity)              (entity or plain value)
```

**AEC example triples:**
```
<Building_A> <rec:hasPart>      <Floor_1>
<Floor_1>    <rec:hasPart>      <Apartment_3B>
<Apartment_3B> <rec:hasPart>   <Room_Kitchen>
<Room_Kitchen> <bot:adjacentZone> <Room_LivingRoom>
```

### Namespaces and Ontologies

- A **Namespace** groups related URIs (like a Python module)
- An **Ontology** defines the vocabulary: what types of entities exist and what relations are valid
- This notebook uses the **Real Estate Core (REC)** ontology and the **Building Topology Ontology (BOT)**

In [ ]:
# Build a tiny dummy Knowledge Graph with rdflib
g = Graph()   # rdflib Graph, not networkx Graph

# Namespaces act like prefixes: base:Mary means https://mydomainname.org/resources/Mary
base = Namespace("https://mydomainname.org/resources/")
ont  = Namespace("https://somepublicontology.org/concepts#")
g.bind("base", base)
g.bind("ont", ont)

# Add triples: (subject, predicate, object)
# URIRef = a URI that identifies an entity
# Literal = a plain value (number, string)
g.add((URIRef("Mary", base), ont["knows"],   URIRef("John", base)))
g.add((URIRef("John", base), ont["knows"],   URIRef("Mary", base)))
g.add((URIRef("Mary", base), ont["age"],     Literal(25)))
g.add((URIRef("John", base), ont["age"],     Literal(25)))
g.add((URIRef("Mary", base), ont["type"],    ont["Human"]))
g.add((URIRef("John", base), ont["type"],    ont["Human"]))
g.add((ont["Human"],         ont["subclass"], ont["BiologicalOrganism"]))

print(f"Total triples in the graph: {len(g)}")
print("\nAll triples:")
for s, p, o in g:
    print(f"  {s.split('/')[-1]:20} {p.split('#')[-1]:15} {str(o).split('/')[-1]}")

### 📌 Instruction: SPARQL — querying a Knowledge Graph

**SPARQL** is the standard query language for RDF graphs — think SQL for Knowledge Graphs.

Basic pattern:
```sparql
SELECT ?subject ?object
WHERE {
    ?subject ont:knows ?object .
}
```

The `?` prefix marks a variable. The query finds all triples matching the pattern and returns bindings.

In [ ]:
# SPARQL query: find who knows whom
results = g.query("""
    PREFIX ont: <https://somepublicontology.org/concepts#>
    SELECT ?person ?friend
    WHERE {
        ?person ont:knows ?friend .
    }
""")

print("SPARQL results — knows relationships:")
for row in results:
    person = str(row.person).split('/')[-1]
    friend = str(row.friend).split('/')[-1]
    print(f"  {person} knows {friend}")

---
## Part 4 — Building the Swiss Dwellings Knowledge Graph

### 📌 Instruction: The AEC dataset

The **Swiss Dwellings** dataset contains ~45,000 apartment models in Switzerland:

- Each apartment has multiple rooms with polygon geometry
- Rooms have a usage type (kitchen, bedroom, bathroom, etc.)
- Rooms are spatially adjacent to each other

**Why model this as a KG?**
- Data comes from heterogeneous sources (IFC, CSV, PDF)
- A KG can integrate all sources under one consistent schema
- The KG enables interoperability via standard ontologies

### The Ontology Schema

Two ontologies are used:
- **[Real Estate Core (REC)](https://www.realestatecore.io)** — models buildings, floors, zones
- **[Building Topology Ontology (BOT)](https://w3id.org/bot)** — models adjacency (`bot:adjacentZone`)

The schema below defines the type of nodes and edges used:

In [ ]:
import networkx as nx
from functools import reduce

# Schema: list of [SourceType, Relation, TargetType]
# This maps directly to the ontology classes and properties
schema = [
    ["rec:Site",      "rec:hasPart",       "rec:Building"],   # A site contains buildings
    ["rec:Building",  "rec:hasPart",       "rec:Level"],      # A building contains levels (floors)
    ["rec:Level",     "rec:hasPart",       "rec:Zone"],       # A level contains zones (apartments)
    ["rec:Zone",      "rec:hasPart",       "rec:Zone"],       # Zones can nest (sub-zones)
    ["rec:Zone",      "rec:hasPart",       "rec:Room"],       # Zones contain rooms
    ["rec:Room",      "bot:adjacentZone",  "rec:Room"],       # ← THE EDGE WE WANT TO PREDICT
    ["rec:Room",      "rec:geometry",      "rec:Polygon"],    # Rooms have polygon geometry
]

# Build a schema graph for visualization
schema_edges = {(x[0], x[2]): {"type": x[1]} for x in schema}
schema_nodes = list(set(n for x in schema for n in [x[0], x[2]]))
schema_g = nx.DiGraph()
schema_g.add_nodes_from(schema_nodes)
schema_g.add_edges_from(schema_edges)
nx.set_edge_attributes(schema_g, schema_edges)

print("Schema node types:", schema_nodes)
print("Schema edge types:", [x[1] for x in schema])
# In the original notebook this is visualized interactively with plot_graph()

### 📌 Instruction: Loading the pre-processed RDF graph

Converting 45K apartments to RDF manually would take too long in a notebook. The original notebook provides:

1. `rdf_graph_no_geom.ttl` — the full KG in Turtle (TTL) format, without polygon geometry nodes (to save RAM)
2. `property_graph.pkl` — a pre-converted NetworkX version of the same graph (easier to traverse in Python)

**Turtle (TTL)** is a compact text format for RDF. Example:
```turtle
@prefix rec: <https://w3id.org/rec#> .
@prefix bot: <https://w3id.org/bot#> .

<urn:uuid:abc123> a rec:Room ;
    bot:adjacentZone <urn:uuid:def456> .
```

**Oxigraph** is a high-performance in-memory RDF store used as a backend for faster SPARQL queries.

In [ ]:
# Load the RDF graph (requires the data files to be present)
# Run download_file() from the original notebook first, or obtain the files manually.

# rdf = Graph(store="Oxigraph")        # use fast in-memory store
# rdf.parse("data/rdf_graph_no_geom.ttl")
# print(f"Triples in graph: {len(rdf)}")

# Example SPARQL query: list all Site entities
# q_site = rdf.query('''
#     PREFIX rec: <https://w3id.org/rec#>
#     SELECT ?site WHERE { ?site a rec:Site . }
# ''')
# for row in q_site:
#     print(row.site)

print("[Stub — uncomment after downloading the dataset files]")

---
## Part 5 — Graph Neural Networks

### 📌 Instruction: Why GNNs for AEC graphs?

Traditional ML (CNN, MLP) works on grid-like data (images, tabular). Graphs are **irregular** — nodes have different numbers of neighbors, there is no fixed ordering, and structure carries meaning.

**Graph Neural Networks (GNNs)** learn node representations by iteratively aggregating information from neighbors:

$$h_i^{(l+1)} = \phi\left( h_i^{(l)},\ \bigoplus_{j \in \mathcal{N}(i)} \psi\left(h_i^{(l)}, h_j^{(l)}\right) \right)$$

- $h_i^{(l)}$ = feature vector of node $i$ at layer $l$
- $\mathcal{N}(i)$ = neighbors of node $i$
- $\psi$ = message function (what to send from neighbor to node)
- $\bigoplus$ = aggregation (sum, mean, max — must be order-invariant)
- $\phi$ = update function (combines own state + aggregated messages)

After $L$ layers, each node's embedding encodes information from its **$L$-hop neighborhood**.

### The Task: Link Prediction

**Problem:** We have a floor plan with rooms but no adjacency edges. Can the model predict which rooms are adjacent?

**Framing:** Given all possible pairs of rooms $(r_i, r_j)$, predict whether `bot:adjacentZone(r_i, r_j)` should exist. This is a **binary classification** problem on edges.

**Model used: GraphSAGE** — samples a fixed-size neighborhood at each layer, making it scalable to large graphs.

### 📌 Instruction: Understanding the heterogeneous GNN architecture

The model has three components:

```
Input features
     │
     ▼
Linear projection   ←── per node type (room, polygon, zone, ...)
+ Embedding lookup  ←── learnable ID embedding
     │
     ▼
GraphSAGE conv ×2   ←── message passing over heterogeneous edges
(to_hetero wraps    ←── this creates separate weights per edge type
 this for us)
     │
     ▼  (room embeddings only)
Classifier MLP      ←── takes [emb_room_i | emb_room_j] → scalar score
     │
     ▼
Binary cross-entropy loss  ←── adjacent=1, not adjacent=0
```

The `to_hetero()` wrapper from PyTorch Geometric automatically replicates the homogeneous GNN across all edge types in the heterogeneous graph.

In [ ]:
# GNN model definition — closely follows the original notebook

class GNN(torch.nn.Module):
    """Two-layer GraphSAGE backbone."""
    def __init__(self, hidden_channels):
        super().__init__()
        # SAGEConv: samples and aggregates neighbor features
        self.conv1 = SAGEConv(hidden_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))  # layer 1: aggregate 1-hop neighbors
        x = self.conv2(x, edge_index)           # layer 2: aggregate 2-hop neighbors
        return x


class Classifier(torch.nn.Module):
    """MLP that scores a pair of room embeddings."""
    def __init__(self, hidden_channels, dropout=0.2):
        super().__init__()
        self.lin1 = torch.nn.Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, 1)  # scalar output
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x_room, edge_label_index):
        # Concatenate embeddings of the two candidate rooms
        z = torch.cat([x_room[edge_label_index[0]],
                        x_room[edge_label_index[1]]], dim=-1)
        z = self.lin1(z).relu()
        z = self.dropout(z)
        z = self.lin2(z)
        return z.view(-1)  # flatten to 1D


class Model(torch.nn.Module):
    """Full heterogeneous GNN model."""
    def __init__(self, hidden_channels, shapes, metadata):
        super().__init__()
        # One linear + embedding per node type
        for key, val in shapes.items():
            setattr(self, key + "_lin", torch.nn.Linear(val[-1], hidden_channels))
            setattr(self, key + "_emb", torch.nn.Embedding(val[0], hidden_channels))
        self.metadata = metadata
        self.gnn = to_hetero(GNN(hidden_channels), metadata=metadata)  # heterogeneous wrapper
        self.classifier = Classifier(hidden_channels=hidden_channels)

    def forward(self, data):
        # Build per-type feature vectors: linear(features) + embedding(node_id)
        x_dict = {}
        for t in self.metadata[0]:
            lin = getattr(self, t + "_lin")
            emb = getattr(self, t + "_emb")
            x_dict[t] = lin(data[t].x) + emb(data[t].node_id)
        # Message passing
        x_dict = self.gnn(x_dict, data.edge_index_dict)
        # Predict adjacency for room pairs
        pred = self.classifier(
            x_dict["room"],
            data["room", "adjacentzone", "room"].edge_label_index
        )
        return pred

print("Model classes defined.")
# To instantiate: model = Model(64, type_shapes, train_data.metadata())
# type_shapes is a dict mapping node type → (num_nodes, feature_dim)

### 📌 Instruction: Training the model

The training loop follows the standard PyTorch pattern:

1. **Batch** the graph using `LinkNeighborLoader` — this samples subgraphs around target edges, making training scalable
2. **Forward pass** — predict scores for all room pairs in the mini-batch
3. **Loss** — binary cross-entropy: `loss = BCE(predictions, ground_truth_labels)`
4. **Backward** — compute gradients
5. **Step** — update weights with Adam optimizer

**Negative sampling:** Not all pairs of rooms are adjacent. The dataset includes both positive edges (truly adjacent rooms) and *negative* edges (rooms in the same apartment that are *not* adjacent). The model must learn to distinguish them.

In [ ]:
# Training loop (requires train_data and model to be instantiated)
# Uncomment after loading the dataset

# optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
# for epoch in range(1, 3):
#     total_loss = total_examples = 0
#     for sampled_data in tqdm.tqdm(train_loader):
#         optimizer.zero_grad()                                # reset gradients
#         sampled_data.to(device)
#         pred = model(sampled_data)                          # forward pass
#         ground_truth = sampled_data["room", "adjacentzone", "room"].edge_label
#         loss = F.binary_cross_entropy_with_logits(pred, ground_truth)
#         loss.backward()                                      # compute gradients
#         optimizer.step()                                     # update weights
#         total_loss += float(loss) * pred.numel()
#         total_examples += pred.numel()
#     print(f"Epoch {epoch:03d} | Loss: {total_loss / total_examples:.4f}")

print("[Stub — uncomment after loading dataset and instantiating model]")

### 📌 Instruction: Evaluating with AUC-ROC

The model is evaluated using **AUC-ROC** (Area Under the Receiver Operating Characteristic curve):

- **AUC = 0.5** → random guessing (model has learned nothing)
- **AUC = 1.0** → perfect classification
- **AUC > 0.85** → generally considered good for link prediction

AUC is preferred over accuracy here because the dataset is **imbalanced** — in a room with 5 neighbors, there are only 5 positive edges but potentially dozens of negative ones.

In [ ]:
# Evaluation loop (requires val_data and model)
# Uncomment after loading the dataset

# preds, ground_truths = [], []
# for sampled_data in tqdm.tqdm(val_loader):
#     with torch.no_grad():                                  # no gradient needed during eval
#         sampled_data.to(device)
#         preds.append(model(sampled_data))
#         ground_truths.append(sampled_data["room", "adjacentzone", "room"].edge_label)
#
# pred = torch.cat(preds, dim=0).cpu().numpy()
# ground_truth = torch.cat(ground_truths, dim=0).cpu().numpy()
# auc = roc_auc_score(ground_truth, pred)
# print(f"Validation AUC: {auc:.4f}")

print("[Stub — uncomment after loading dataset]")

---
## Part 6 — Inference: Predicting Room Adjacency

### 📌 Instruction: How inference works

At inference time:

1. Take a floor plan with rooms but **no adjacency edges**
2. Generate **all possible pairs** of rooms within the apartment (complete graph)
3. Feed them to the model → get a probability score for each pair
4. Keep pairs above a **threshold** (e.g., 0.6) as predicted adjacencies

**Threshold sensitivity:** A higher threshold = more conservative predictions (fewer but more confident edges). The threshold should be tuned for the use case (e.g., false negatives more costly in fire safety planning than false positives).

The `infer_edges` function below does steps 2–4:

In [ ]:
def infer_edges(data, test_unit_id, threshold=0.5):
    """
    Given a HeteroData object and an apartment ID,
    predict which room pairs are adjacent.

    Returns an array of shape (2, N) with N predicted edge pairs.
    """
    s, e, t = "room", "adjacentzone", "room"
    data = copy(data)

    # Get all room node IDs belonging to this apartment
    test_unit_rooms = data['room'].node_id[
        data['room'].unit_id == test_unit_id
    ].cpu().numpy()

    # Generate all possible (room_i, room_j) pairs — complete graph
    test_unit_rooms_edges = list(combinations(test_unit_rooms, 2))
    test_unit_rooms_edges = torch.Tensor(test_unit_rooms_edges).T.long()

    # Replace edge_label_index with candidate pairs
    data[s, e, t].edge_label_index = test_unit_rooms_edges
    data = data.to('cpu')  # use 'cuda' if available

    with torch.no_grad():
        # sigmoid converts raw logit → probability in [0, 1]
        new_pred = torch.sigmoid(model(data)).cpu().numpy().round(3)

    # Return only pairs above the threshold
    res = test_unit_rooms_edges[:, new_pred >= threshold].T.numpy()
    return res

print("infer_edges() defined.")

### 📌 Instruction: Visualizing predictions vs ground truth

After inference:
- **Test graph (input):** rooms as nodes, NO adjacency edges
- **Predicted graph:** test graph + edges predicted by the model
- **Ground truth graph:** original graph with all real `bot:adjacentZone` edges

By comparing predicted vs ground truth visually (and numerically via AUC), we assess how well the model has learned the spatial layout rules of apartments.

In [ ]:
# Inference + visualization (requires test_data and trained model)
# Uncomment after loading the dataset

# threshold = 0.6
#
# # Remove existing adjacency edges from the test graph
# test_graph = actual_graph.copy().to_undirected()
# edges_to_remove = [
#     e for e in test_graph.edges()
#     if test_graph.edges[e]['type'] == "bot:adjacentZone"
# ]
# test_graph.remove_edges_from(edges_to_remove)
#
# # Run inference
# infered_edges = infer_edges(new_data, test_unit, threshold=threshold)
#
# # Map integer node IDs back to URIs and add edges to graph
# infered_edges_uris = {}
# for n1, n2 in infered_edges:
#     uri1 = id_mapping["rooms"][n1]
#     uri2 = id_mapping["rooms"][n2]
#     infered_edges_uris[(uri1, uri2)] = {"type": "bot:adjacentZone"}
#
# test_graph.add_edges_from(infered_edges_uris)
# nx.set_edge_attributes(test_graph, infered_edges_uris)
#
# # plot_graph(test_graph, ...)   # shows predicted adjacencies
# # plot_graph(actual_graph, ...) # shows ground truth

print("[Stub — uncomment after loading dataset and trained model]")

---
## Part 7 — Graph RAG (Work in Progress)

### 📌 Instruction: What is Graph RAG?

**Graph RAG** (Retrieval-Augmented Generation over a Knowledge Graph) combines:
- A **Large Language Model (LLM)** (here: OpenAI GPT) for natural language understanding
- A **Knowledge Graph** for structured, factual data

Workflow:
```
User question (natural language)
        │
        ▼
  LLM generates SPARQL query
        │
        ▼
  SPARQL runs on the KG
        │
        ▼
  Results returned to LLM
        │
        ▼
  LLM formulates natural language answer
```

**AEC application examples:**
- *"How many kitchens are in building X?"*
- *"Which floors have the most apartment units?"*
- *"What is the average number of rooms per apartment?"*

> **Note:** This section is marked WIP in the original notebook. The LLM-generated SPARQL may be sub-optimal or time out on complex queries.

In [ ]:
# Graph RAG: LLM-powered natural language querying of the KG
# Requires: OpenAI API key + rdf graph loaded

# Example questions you could ask:
questions = [
    "What is the number of buildings in the dataset?",
    "What is the number of 'KITCHEN' rooms in the dataset?",
    "How many 'KITCHEN' rooms are on each building on average?",
]

# The ask_kg() function from the original notebook:
# 1. Sends the question + KG schema to the LLM
# 2. LLM generates a SPARQL query
# 3. Runs SPARQL on the rdf graph
# 4. Returns the result or retries if the query fails

# answer = ask_kg(questions[0], rdf, iterate=True, verbose=True)
# print(answer)

print("Example questions:")
for q in questions:
    print(f"  - {q}")
print("\n[Requires: openai API key + rdf loaded — uncomment to run]")

---
## Summary & Key Takeaways

### 📌 What this notebook demonstrates

| Concept | What was shown | AEC relevance |
|---------|---------------|---------------|
| **Graph theory** | Nodes, edges, adjacency matrices, graph types | Any relational AEC data |
| **Heterogeneous graphs** | Multiple node/edge types | IFC objects (walls, rooms, sensors) |
| **Knowledge Graphs** | RDF triples, URIs, SPARQL | Data integration from IFC, PDF, sensors |
| **Ontologies (REC, BOT)** | Shared vocabulary for AEC entities | Interoperability between software tools |
| **Swiss Dwellings dataset** | 45K apartments with geometry | Benchmark for spatial ML research |
| **GraphSAGE** | Message passing on heterogeneous graphs | Any graph-structured prediction task |
| **Link prediction** | Predicting room adjacencies | Completing incomplete BIM data |
| **Graph RAG** | LLM + SPARQL for NL querying | Querying BIM/FM data conversationally |

### Next steps for your own research

1. **Extend the schema** — add sensors, MEP systems, or material properties as new node types
2. **Use your own IFC** — parse IFC files with `ifcopenshell`, convert to RDF using `ifcowl`, build your KG
3. **Try other GNN tasks** — node classification (predict room type from geometry), graph regression (predict energy performance)
4. **Improve Graph RAG** — fine-tune the LLM on AEC SPARQL patterns, add schema-aware prompting

### Further reading

- [PyTorch Geometric docs](https://pytorch-geometric.readthedocs.io)
- [CS224W: Machine Learning with Graphs (Stanford)](http://web.stanford.edu/class/cs224w/)
- [Real Estate Core ontology](https://www.realestatecore.io)
- [Building Topology Ontology (BOT)](https://w3id.org/bot)
- [GraphSAGE paper](https://arxiv.org/abs/1706.02216)
- [Swiss Dwellings dataset](https://doi.org/10.5281/zenodo.7788422)